## Personal Career Agent

This is a personal career agent that responds to all your questions on me and if the response is not available then it sends me a notification with the question so that I can update the knowledge base. It allows for storing the questions whose answers were not available in  relational database until they are responded to by me.

Deployed version: https://huggingface.co/spaces/johnmboga/johnerick-personal-career-agent

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr
from utils.db import DatabaseUtils
from utils.ingest import DocumentIngester
from config import Config
from pydantic import BaseModel

In [2]:
# Create a Pydantic model for the Evaluation
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

In [4]:
#create database connection
db = DatabaseUtils()
cfg = Config()  # optional, you can pass your own config

In [5]:
# Use the same Chroma collection as chat (single source of truth from Config)
collection = cfg.career_collection

In [7]:

ingester = DocumentIngester(config=cfg, docs_folder="docs", chunk_size=500)
count = ingester.ingest()
print(f"Ingestion complete! Ingested {count} document(s).")

Ingestion complete! Ingested 5 document(s).


In [8]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [9]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
def insert_unknown_question(question, user_id, notes = None):
    db.insert_unknown_question(question, user_id, notes)
    

In [41]:
insert_unknown_question("What is your current salary?", "user123","Current salary is not available")

In [11]:
def get_unknown_questions():
    return db.get_unknown_questions()

In [10]:
get_unknown_questions()

[(3,
  'What is your current salary?',
  'user123',
  '2026-03-16T15:41:09.178434',
  'Current salary is not available',
  0),
 (4,
  'What is your current salary?',
  'user123',
  '2026-03-16T16:27:31.534791',
  'Current salary is not available',
  0)]

In [12]:
def mark_as_answered(question_id):
    db.mark_as_answered(question_id)

In [12]:
mark_as_answered(2)

In [13]:
def record_unknown_question(question, user_id=None, notes=None):
    insert_unknown_question(question, user_id, notes)
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [14]:
record_unknown_question("What is your current salary?", "user123","Current salary is not available")

Push: Recording What is your current salary? asked that I couldn't answer


{'recorded': 'ok'}

In [14]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Use this tool to record a question that the system cannot answer and send a push notification to the admin for follow-up",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that the system was unable to answer"
            },
            "user_id": {
                "type": "string",
                "description": "Identifier of the user who asked the question, if available"
            },
            "notes": {
                "type": "string",
                "description": "Optional context or metadata about the conversation"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [15]:
tools = [{"type": "function", "function": record_unknown_question_json}]

In [17]:
tools

[{'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Use this tool to record a question that the system cannot answer and send a push notification to the admin for follow-up',
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': 'The question that the system was unable to answer'},
     'user_id': {'type': 'string',
      'description': 'Identifier of the user who asked the question, if available'},
     'notes': {'type': 'string',
      'description': 'Optional context or metadata about the conversation'}},
    'required': ['question'],
    'additionalProperties': False}}}]

In [16]:

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        if tool:
            allowed = {"question", "user_id", "notes"}
            kwargs = {k: v for k, v in arguments.items() if k in allowed}
            result = tool(**kwargs)
        else:
            result = {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [17]:
name = "John Mboga"

In [18]:
# Template: {name} and {retrieved_context} are filled when building the prompt
system_prompt_base = """
You are acting as {name}, a senior software engineer. Always answer as if you are {name}.
Do NOT provide generic responses. Only provide information that is:
- retrieved from the provided context
- or previously answered questions stored in the vector database
- If you truly don't know, politely state that the information is not available and record the question using the record_unknown_question tool.

You are professional, confident, and informative.
Always make your answers concise and directly relevant to the question.

## Context for this turn:
{retrieved_context}

Now answer the user's question below.
"""

In [19]:
def retrieve_context(question):
    """Retrieve relevant chunks from the career collection for the given question."""
    top_k = 5
    query_embedding = cfg.openai.embeddings.create(
        model="text-embedding-3-large",
        input=[question]
    ).data[0].embedding
    results = cfg.career_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    chunks = [item for sublist in results["documents"] for item in sublist]
    return "\n\n".join(chunks) if chunks else ""

In [20]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [21]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [22]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.chat.completions.parse(model="google/gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [23]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt_base + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [24]:
def add_current_response(message, assistant_reply):
    """Add the current user message and assistant reply to the vector database."""
    documents = [message, assistant_reply]  
    metadatas = [{"role": "user"}, {"role": "assistant"}]
    embeddings = cfg.openai.embeddings.create(
        model="text-embedding-3-large",
        input=documents
    ).data
    cfg.career_collection.add(
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings,
    )

In [25]:
def get_messages(message, history):
    """
    message: current user message
    history: list of {"role": "user"/"assistant", "content": str}
    cfg: Config object with career_collection & openai client
    """
    # Step 1: Retrieve relevant context from Chroma
    context_text = retrieve_context(message)
    print(f"Retrieved context: {context_text}")

    # Step 2: Construct system prompt with retrieved context
    system_prompt_with_context = system_prompt_base.format(
        name=name,
        retrieved_context=context_text or "(No relevant context found.)"
    )

    # Step 4: Build messages for LLM (system + history + current user)
    messages = [{"role": "system", "content": system_prompt_with_context}] + history + [{"role": "user", "content": message}]
    return messages

In [26]:
def chat(message, history):
    """
    message: current user message
    history: list of {"role": "user"/"assistant", "content": str}
    cfg: Config object with career_collection & openai client
    """
    messages = get_messages(message, history)

    done = False
    while not done:
        response = cfg.openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools  # if you’re using tool-calling
        )

        finish_reason = response.choices[0].finish_reason
        reply = response.choices[0].message.content
        evaluation = evaluate(reply, message, history)
        if evaluation.is_acceptable:
            print("Passed evaluation - checking for tool calls to return reply")
            if finish_reason == "tool_calls":
                assistant_message = response.choices[0].message
                tool_calls = assistant_message.tool_calls
                results = handle_tool_calls(tool_calls)
                messages.append(assistant_message)
                messages.extend(results)
            else:
                done = True
            
    
            # add_current_response(message, reply)
        else:
            print("Failed evaluation - retrying")
            print(evaluation.feedback)
            reply = rerun(reply, message, history, evaluation.feedback)

            

    

    return reply or ""

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Retrieved context: Boot, Angular, JSF, ADF). ● Mentored engineers, handled hiring, and improved engineering culture. Felsoft Systems — Senior Software Engineer Jun 2014 – Apr 2015 ● Delivered platforms for NGOs (PIMS, AAHI) and established early Agile practices. PwC — Technology Consultant Sep 2013 – May 2014 ● Led data migration for a major Kenyan bank, reducing migration time by 30% via custom tooling. SELECTED PROJECTS ● Encrypted P2P Support Chat — Hyperswarm + Hypercore + crypto + human handoff + AI assistants ● AML Rules Engine — Configurable Node.js rule evaluation with ACID guarantees ● BioID System — Identity verification with SSO + liveness detection (React, Node.js, AWS) ● Internal React/TS component frameworks for onboarding, rules engines, search, and workflow UIs EDUCATION B.Sc. Computer Science — University of Nairobi (2009–2013) WHY I’M A FIT FOR DEEL ● 11+ years building full-stack systems with TypeScript, React, Node.js, Express/NestJS, PostgreSQL ● Experience with 24

## And now for deployment

This code is in `app.py`

We will deploy to HuggingFace Spaces.

Before you start: remember to update the files in the "me" directory - your LinkedIn profile and summary.txt - so that it talks about you! Also change `self.name = "Ed Donner"` in `app.py`..  

Also check that there's no README file within the 1_foundations directory. If there is one, please delete it. The deploy process creates a new README file in this directory for you.

1. Visit https://huggingface.co and set up an account  
2. From the Avatar menu on the top right, choose Access Tokens. Choose "Create New Token". Give it WRITE permissions - it needs to have WRITE permissions! Keep a record of your new key.  
3. In the Terminal, run: `uv tool install 'huggingface_hub[cli]'` to install the HuggingFace tool, then `hf auth login --token YOUR_TOKEN_HERE`, like `hf auth login --token hf_xxxxxx`, to login at the command line with your key. Afterwards, run `hf auth whoami` to check you're logged in  
4. Take your new token and add it to your .env file: `HF_TOKEN=hf_xxx` for the future
5. From the 1_foundations folder, enter: `uv run gradio deploy` 
6. Follow its instructions: name it "career_conversation", specify app.py, choose cpu-basic as the hardware, say Yes to needing to supply secrets, provide your openai api key, your pushover user and token, and say "no" to github actions.  

Thank you Robert, James, Martins, Andras and Priya for these tips.  
Please read the next 2 sections - how to change your Secrets, and how to redeploy your Space (you may need to delete the README.md that gets created in this 1_foundations directory).

#### More about these secrets:

If you're confused by what's going on with these secrets: it just wants you to enter the key name and value for each of your secrets -- so you would enter:  
`OPENAI_API_KEY`  
Followed by:  
`sk-proj-...`  

And if you don't want to set secrets this way, or something goes wrong with it, it's no problem - you can change your secrets later:  
1. Log in to HuggingFace website  
2. Go to your profile screen via the Avatar menu on the top right  
3. Select the Space you deployed  
4. Click on the Settings wheel on the top right  
5. You can scroll down to change your secrets (Variables and Secrets section), delete the space, etc.

#### And now you should be deployed!

If you want to completely replace everything and start again with your keys, you may need to delete the README.md that got created in this 1_foundations folder.

Here is mine: https://huggingface.co/spaces/ed-donner/Career_Conversation

I just got a push notification that a student asked me how they can become President of their country 😂😂

For more information on deployment:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

To delete your Space in the future:  
1. Log in to HuggingFace
2. From the Avatar menu, select your profile
3. Click on the Space itself and select the settings wheel on the top right
4. Scroll to the Delete section at the bottom
5. ALSO: delete the README file that Gradio may have created inside this 1_foundations folder (otherwise it won't ask you the questions the next time you do a gradio deploy)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• First and foremost, deploy this for yourself! It's a real, valuable tool - the future resume..<br/>
            • Next, improve the resources - add better context about yourself. If you know RAG, then add a knowledge base about you.<br/>
            • Add in more tools! You could have a SQL database with common Q&A that the LLM could read and write from?<br/>
            • Bring in the Evaluator from the last lab, and add other Agentic patterns.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your career alter-ego) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>